In [ ]:
'''
在该Python文件相同目录下创建xxx文件夹，所有文件放在里面
最终会在相同目录生成的excel文件
'''
import os
import sys
import docx #pip  install python-docx
import shutil
import pandas as pd
from doc2docx import convert

os.chdir('xxx') #进入子文件夹xxx，里面存放所有文件
current_path = os.getcwd() #获取当前路径
files=os.listdir() # 列出当前文件夹所有文件名

# 创建document文件夹用来放转换后的文件
exist=os.path.exists('document')
if not exist:
    os.mkdir('document')
# 创建error文件夹，无法正常读取的文件会复制到里面
exist=os.path.exists('error')
if not exist:
    os.mkdir('error')
os.chdir(sys.path[0]) #回到初始目录
for file in files:
    # 筛选出doc格式文件，转化为docx文件，如果是docx文件直接复制
    dot_pos = file.find('.')
    extension = file[dot_pos:]
    filename = file[:dot_pos]
    if extension == '.doc' or extension == '.docx':
        original_path = os.path.join(current_path,file)
        new_file = filename + ".docx"
        new_path = os.path.join(current_path,'document',new_file)
        convert(original_path, new_path)

In [ ]:
os.chdir(os.path.join('xxx','document')) #进入子文件夹xxx，里面存放所有docx文件
current_path = os.getcwd() #获取当前路径
files=os.listdir() # 列出当前文件夹所有文件名
os.chdir(sys.path[0]) #回到初始目录
df_0 = pd.DataFrame()
for file in files:
    try:
        doc = docx.Document(os.path.join(current_path,file))
        #获取页眉中的编号
        headers = []
        for section in doc.sections:
            headers.append(section.header)
        tables = headers[0].tables
        table = tables[0]
        s = table.rows[0].cells[0].text
        start = s.find(": ")
        n1 = s[start+2:]
        #获取第一个表格中的编号
        tables = doc.tables 
        t = tables[0].rows[0].cells[1].text
        end = t.find(' ')
        n2 = t[:end]
        #获取产品名
        product  = t[end:].strip()
        #获取文件名
        name = file[:file.find('.')]
        df = pd.DataFrame({"ISN No.":[n1], "ID No.":[n2], "Product":[product], "Filename":[name]})
        #df_0 = df_0.append(df, ignore_index=True)
        df_0 = pd.concat([df_0,df],ignore_index=True)
    except:
        #把无法正常读取的文件复制到error文件夹做后续处理
        shutil.copyfile(os.path.join(current_path,file), os.path.join(sys.path[0],'xxx','error',file))
df_0.to_excel(excel_writer='secret.xlsx', encoding='utf-8', index=False)